In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import xgboost as xgb
import catboost as cb
import pickle

# Load data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.columns)
print(test.columns)

# Convert datetime
train['datetime'] = pd.to_datetime(train['datetime'], errors='coerce')
test['datetime'] = pd.to_datetime(test['datetime'], errors='coerce')

# Extract features
for df in [train, test]:
    df['hour'] = df['datetime'].dt.hour
    df['day'] = df['datetime'].dt.day
    df['month'] = df['datetime'].dt.month
    df['year'] = df['datetime'].dt.year
    df['dayofweek'] = df['datetime'].dt.dayofweek

# Drop columns that leak target information
train = train.drop(['datetime', 'casual', 'registered'], axis=1)

# Save datetime for submission
test_datetime = test['datetime']
test = test.drop(['datetime'], axis=1)

# Prepare target and features
y = np.log1p(train['count'])
X = train.drop('count', axis=1)

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Dictionary to store models and their validation scores
models = {}
validation_scores = {}

# 1. Gradient Boosting (Original)
print("\n" + "="*50)
print("Training Gradient Boosting Regressor...")
print("="*50)

gb_model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

gb_model.fit(X_train, y_train)
val_pred_gb = gb_model.predict(X_val)
rmsle_gb = np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(val_pred_gb)))
print(f"Validation RMSLE: {rmsle_gb:.6f}")

models['GradientBoosting'] = gb_model
validation_scores['GradientBoosting'] = rmsle_gb

# 2. Random Forest
print("\n" + "="*50)
print("Training Random Forest Regressor...")
print("="*50)

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=25,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
val_pred_rf = rf_model.predict(X_val)
rmsle_rf = np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(val_pred_rf)))
print(f"Validation RMSLE: {rmsle_rf:.6f}")

models['RandomForest'] = rf_model
validation_scores['RandomForest'] = rmsle_rf

# 3. XGBoost
print("\n" + "="*50)
print("Training XGBoost Regressor...")
print("="*50)

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
val_pred_xgb = xgb_model.predict(X_val)
rmsle_xgb = np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(val_pred_xgb)))
print(f"Validation RMSLE: {rmsle_xgb:.6f}")

models['XGBoost'] = xgb_model
validation_scores['XGBoost'] = rmsle_xgb

# 4. CatBoost
print("\n" + "="*50)
print("Training CatBoost Regressor...")
print("="*50)

cb_model = cb.CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    verbose=100,
    random_seed=42
)

cb_model.fit(X_train, y_train, verbose=False)
val_pred_cb = cb_model.predict(X_val)
rmsle_cb = np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(val_pred_cb)))
print(f"Validation RMSLE: {rmsle_cb:.6f}")

models['CatBoost'] = cb_model
validation_scores['CatBoost'] = rmsle_cb

# Display all validation scores
print("\n" + "="*50)
print("Validation RMSLE Scores Summary")
print("="*50)
for model_name, score in validation_scores.items():
    print(f"{model_name:20s}: {score:.6f}")

# Find best model
best_model_name = min(validation_scores, key=validation_scores.get)
print(f"\nBest Model: {best_model_name} with RMSLE: {validation_scores[best_model_name]:.6f}")

# Retrain best model on full dataset
print(f"\nRetraining {best_model_name} on full dataset...")
best_model = models[best_model_name]
best_model.fit(X, y)

# Save all models
print("\n" + "="*50)
print("Saving models...")
print("="*50)

# Save individual models
for model_name, model in models.items():
    filename = f"bike_model_{model_name.lower()}.pkl"
    with open(filename, 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ {filename} saved")

# Save the best model separately
best_filename = f"bike_model_best_{best_model_name.lower()}.pkl"
with open(best_filename, 'wb') as f:
    pickle.dump(best_model, f)
print(f"\n✓ Best model saved as: {best_filename}")

# Also save all models in a dictionary for easy access
all_models_filename = "bike_models_all.pkl"
with open(all_models_filename, 'wb') as f:
    pickle.dump(models, f)
print(f"✓ All models saved in: {all_models_filename}")

print("\n✅ All models saved successfully!")

# Optional: Create ensemble prediction (simple average)
print("\n" + "="*50)
print("Creating ensemble predictions...")
print("="*50)

# Make predictions on validation set with all models
ensemble_pred_val = np.zeros_like(val_pred_gb)
for model in models.values():
    ensemble_pred_val += model.predict(X_val)
ensemble_pred_val /= len(models)

rmsle_ensemble = np.sqrt(mean_squared_log_error(
    np.expm1(y_val), 
    np.expm1(ensemble_pred_val)
))
print(f"Ensemble (Simple Average) RMSLE: {rmsle_ensemble:.6f}")

# Create ensemble model (will be used for predictions by averaging)
print("\n" + "="*50)
print("Saving ensemble model info...")
print("="*50)

ensemble_info = {
    'models': models,
    'validation_scores': validation_scores,
    'best_model_name': best_model_name,
    'ensemble_rmsle': rmsle_ensemble
}

ensemble_filename = "bike_ensemble_info.pkl"
with open(ensemble_filename, 'wb') as f:
    pickle.dump(ensemble_info, f)
print(f"✓ Ensemble info saved as: {ensemble_filename}")

print("\n✅ All processing complete!")

Index(['datetime', 'season', 'holiday', 'workingday', 'weather', 'temp',
       'atemp', 'humidity', 'windspeed', 'casual', 'registered', 'count'],
      dtype='object')
Index(['datetime', 'season', 'holiday', 'workingday', 'weather', 'temp',
       'atemp', 'humidity', 'windspeed', 'casual', 'registered'],
      dtype='object')

Training Gradient Boosting Regressor...
Validation RMSLE: 0.282390

Training Random Forest Regressor...
Validation RMSLE: 0.439985

Training XGBoost Regressor...
Validation RMSLE: 0.277318

Training CatBoost Regressor...
Validation RMSLE: 0.282540

Validation RMSLE Scores Summary
GradientBoosting    : 0.282390
RandomForest        : 0.439985
XGBoost             : 0.277318
CatBoost            : 0.282540

Best Model: XGBoost with RMSLE: 0.277318

Retraining XGBoost on full dataset...

Saving models...
✓ bike_model_gradientboosting.pkl saved
✓ bike_model_randomforest.pkl saved
✓ bike_model_xgboost.pkl saved
✓ bike_model_catboost.pkl saved

✓ Best model saved as: b